# Python 기초 통합 프로젝트
## Part 2. 판매 점검 코드를 함수화하고 결과 저장하기

### Part 1과의 연결

Part 1에서 작성한 판매금액 구간 분류와 배송 확인 거래 검색 코드를 함수로 바꾸어 반복 사용이 가능한 구조로 개선합니다.

### 실무 시나리오

영업관리팀은 매일 새로운 판매 데이터에 같은 점검 기준을 적용합니다. 데이터 담당자는 반복 코드를 함수로 정리하고, 배송 확인 대상 거래를 CSV 파일로 저장하여 담당자에게 전달해야 합니다.

### 과제 목표

- 반복 코드를 재사용 가능한 함수로 작성할 수 있습니다.
- 매개변수, 반환값, 기본값을 사용할 수 있습니다.
- `TypeError`, `ValueError`, `PermissionError` 등 오류 유형을 구분해 처리할 수 있습니다.
- 분석 결과를 CSV 파일로 저장하고 다시 확인할 수 있습니다.
- 심화 문제에서 고액 반품 결과를 JSON으로 저장할 수 있습니다.

### 사용 환경

- 결과물: `.ipynb` 파일 1개
- 필수 생성 파일: `delivery_check_sales.csv`
- 심화 생성 파일: `high_value_return_sales.json`
- 사용 언어: Python 3.X
- 사용 라이브러리: pandas

## 제공 코드. 데이터 불러오기

In [5]:
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

print("전체 거래 수:", len(df))

전체 거래 수: 500


# 문제 1. 판매금액 구간 분류 함수 만들기

## 함수

```python
classify_sales_by_amount(sale_ids, sale_amounts)
```

## 요구사항

1. 두 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 두 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 데이터가 비어 있으면 `ValueError`를 발생시킵니다.
4. 고가·중가·일반 거래의 `SaleID`를 딕셔너리로 반환합니다.
5. 함수를 호출하고 결과를 출력합니다.

In [11]:
# 문제 1 코드를 작성하세요.

def classify_sales_by_amount(sale_ids, sale_amounts):


        if not isinstance(sale_ids,list) or not isinstance(sale_amounts,list): 
                raise TypeError
        if len(sale_ids) != len(sale_amounts):
                raise ValueError

        if not sale_ids or not sale_amounts:
                raise ValueError
        

        result = {"고가":[], "중가":[],"일반":[]}


        for i in range(len(sale_amounts)): 

                sale_amount = sale_amounts[i]
                sale_id = sale_ids[i]
                                
                if sale_amount >= 70000000:

                        result["고가"].append(sale_id)               

                elif sale_amount >= 40000000: 

                        result["중가"].append(sale_id)

                else:
                        result["일반"].append(sale_id)

        return result



result = classify_sales_by_amount(sale_ids, sale_amounts)
print(result)





{'고가': ['S2025010064', 'S2025010264', 'S2025010093', 'S2025010135', 'S2025010483', 'S2025010165', 'S2025020115', 'S2025020161', 'S2025020030', 'S2025020413', 'S2025020437', 'S2025020031', 'S2025030414', 'S2025030139', 'S2025030005', 'S2025030049', 'S2025040389', 'S2025040494', 'S2025040336', 'S2025040344', 'S2025040284', 'S2025040369', 'S2025040429', 'S2025040262', 'S2025040288', 'S2025040206', 'S2025040178', 'S2025050451', 'S2025050444', 'S2025050127', 'S2025050303', 'S2025050183', 'S2025050260', 'S2025050273', 'S2025050401', 'S2025060015', 'S2025060430', 'S2025060153', 'S2025060072', 'S2025060189', 'S2025060217', 'S2025060331', 'S2025060490', 'S2025070458', 'S2025070220', 'S2025070073', 'S2025070474', 'S2025070083', 'S2025080218', 'S2025080209', 'S2025080212', 'S2025080439', 'S2025080108', 'S2025090089', 'S2025090254', 'S2025090074', 'S2025100133', 'S2025100176', 'S2025100316', 'S2025100164', 'S2025100392', 'S2025100479', 'S2025100227', 'S2025110248', 'S2025110197', 'S2025110351', 'S

# 문제 2. 배송 확인 거래 검색 함수 만들기

## 함수

```python
find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
)
```

## 요구사항

1. 세 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 세 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 배송일 20일 이상이면서 출고완료가 아닌 거래의 `SaleID`를 반환합니다.
4. 정상 입력과 잘못된 입력을 각각 테스트합니다.
5. `TypeError`와 `ValueError`를 별도의 `except`에서 처리합니다.

In [25]:
# 문제 2 코드를 작성하세요.

def find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
):

    if not isinstance(sale_ids, list) or not isinstance(delivery_days, list) or not isinstance(stock_statuses, list):
        raise TypeError
    if len(sale_ids) != len(delivery_days) != len(stock_statuses):
        raise ValueError


    result = []

    for i in range(len(sale_ids)):
        if delivery_days[i] >= 20 and stock_statuses[i] != "출고 완료":
            result.append(sale_ids[i])

    return result

sale_ids = ["A001", "A002", "A003"]
delivery_days = [10, 25, 30]
stock_statuses = ["출고 완료", "배송 중", "배송 준비"]

print(find_delivery_check_sales(sale_ids, delivery_days, stock_statuses))

try:
    find_delivery_check_sales(
        "A001",
        [10, 25],
        ["출고 완료", "배송 중"]
    )
except TypeError:
    print("TypeError 발생")



   

['A002', 'A003']
TypeError 발생


# 문제 3. 배송 점검 결과를 CSV로 저장하기

## 요구사항

1. 문제 2에서 반환된 `SaleID`로 원본 DataFrame의 거래를 선택합니다.
2. `delivery_check_sales.csv`로 저장합니다.
3. 파일 인덱스는 저장하지 않습니다.
4. 저장 파일을 다시 불러와 거래 수를 비교합니다.
5. `PermissionError`와 그 외 `OSError`를 구분하여 처리합니다.

In [28]:
# 문제 3 코드를 작성하세요.

delivery_check_ids = find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
)

try:
    delivery_check_df = df[
        df["SaleID"].isin(delivery_check_ids)
    ]

    delivery_check_df.to_csv(
        "delivery_check_sales.csv",
        index=False
    )

    saved_df = pd.read_csv("delivery_check_sales.csv")

    print("원본에서 선택된 거래 수:", len(delivery_check_df))
    print("저장 후 불러온 거래 수:", len(saved_df))

except PermissionError:
    print("파일에 접근할 권한이 없습니다.")

except OSError:
    print("파일 저장 또는 불러오기에 문제가 발생했습니다.")

원본에서 선택된 거래 수: 0
저장 후 불러온 거래 수: 0


# 심화 문제. 고액 반품 검색 및 JSON 저장

이 문제는 **5점 수준을 위한 선택 문제**입니다.

## 요구사항

1. `find_high_value_returns()` 함수를 작성합니다.
2. `min_amount=70_000_000` 기본값을 사용합니다.
3. 입력 자료형이 잘못되면 `TypeError`, 기준값이 잘못되면 `ValueError`를 발생시킵니다.
4. 고액 반품 거래를 JSON 파일로 저장합니다.
5. 저장한 JSON을 다시 읽어 거래 수를 확인합니다.
6. `PermissionError`, `OSError`, `ValueError`를 구분해 처리합니다.

In [ ]:
# 심화 문제 코드를 작성하세요.

# 제출 결과물

| 결과물 | 구분 |
|---|---|
| 판매금액 구간 분류 함수 | 필수 |
| 배송 확인 거래 함수 | 필수 |
| `TypeError`, `ValueError` 구분 처리 | 필수 |
| 배송 확인 CSV 저장 및 재확인 | 필수 |
| `PermissionError`, `OSError` 구분 처리 | 필수 |
| 고액 반품 함수와 JSON 저장 | 심화 |